# Gaussian SLDS with high-dimensional emissions

This example keeps the latent dynamics two-dimensional so they remain easy to inspect,
while embedding the latent trajectory into an eight-dimensional Gaussian observation
space. The emission loading matrix has orthonormal columns, so the embedding preserves
latent geometry while still requiring the model to recover a low-dimensional latent
representation from higher-dimensional observations.


In [ ]:
%matplotlib inline

import _plotting as plot
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from xxm.core.align import align_procrustes as align_latent
from xxm.core.align import match_states_to_true as match_states
from xxm.slds import GaussianSLDS

In [ ]:
NUM_STATES = 3
LATENT_DIM = 2
OBSERVATION_DIM = 8
NUM_STEPS = 2_000


def make_emission_coefficients(observation_dim: int) -> jax.Array:
    """Embed 2D latents isometrically in a higher-dimensional observation space."""
    angles = 2 * jnp.pi * jnp.arange(observation_dim) / observation_dim

    coefficients = jnp.stack(
        [
            jnp.cos(angles),
            jnp.sin(angles),
        ],
        axis=1,
    )

    return jnp.sqrt(2 / observation_dim) * coefficients


def make_true_model() -> GaussianSLDS:
    emission_coefficients = make_emission_coefficients(OBSERVATION_DIM)

    return GaussianSLDS.from_params(
        initial_probs=jnp.array([0.4, 0.3, 0.3]),
        transition_probs=jnp.array(
            [
                [0.97, 0.02, 0.01],
                [0.02, 0.97, 0.01],
                [0.02, 0.02, 0.96],
            ]
        ),
        latent_initial_means=jnp.zeros((NUM_STATES, LATENT_DIM)),
        latent_initial_covariances=jnp.tile(
            0.25 * jnp.eye(LATENT_DIM),
            (NUM_STATES, 1, 1),
        ),
        dynamics_coefficients=jnp.array(
            [
                [[0.94, 0.22], [-0.22, 0.94]],
                [[0.94, -0.22], [0.22, 0.94]],
                [[0.98, 0.00], [0.00, 0.70]],
            ]
        ),
        dynamics_bias=jnp.zeros((NUM_STATES, LATENT_DIM)),
        dynamics_covariances=jnp.tile(
            0.03 * jnp.eye(LATENT_DIM),
            (NUM_STATES, 1, 1),
        ),
        emission_coefficients=emission_coefficients,
        emission_bias=jnp.zeros(OBSERVATION_DIM),
        emission_covariance=0.02 * jnp.eye(OBSERVATION_DIM),
    )


true_model = make_true_model()

true_states, true_latents, observations = true_model.sample(
    key=jax.random.key(0),
    num_steps=NUM_STEPS,
)

In [ ]:
def plot_obs(coefficients, observations):

    fig, axs = plt.subplots(
        ncols=2,
        figsize=(10, 3.5),
        constrained_layout=True,
    )

    image = axs[0].imshow(
        np.asarray(coefficients),
        aspect='auto',
    )
    axs[0].set(
        title='Emission coefficients',
        xlabel='latent dimension',
        ylabel='observation dimension',
        xticks=range(LATENT_DIM),
    )
    fig.colorbar(image, ax=axs[0])

    plot.plot_traces_image(
        axs[1],
        observations,
    )
    axs[1].set(title='Observed sequence')


plot.plot_dyn_conditional_linear(true_model.dynamics)

plot_obs(true_model.emissions.affine.coefficients, observations)

In [ ]:
initial_model = GaussianSLDS.from_arhmm(
    key=jax.random.key(1),
    observations=observations,
    num_states=NUM_STATES,
    latent_dim=LATENT_DIM,
    num_arhmm_iters=1000,
)

fit = initial_model.fit(
    observations,
    num_iters=50,
    num_inference_iters=20,
)

learned_model = fit.model


fig, ax = plt.subplots(
    figsize=(6, 3),
    constrained_layout=True,
)

plot.plot_fit_progress(
    fit.objective_trace,
    name='ELBO',
    ax=ax,
)
ax.set(title='Variational EM')

In [ ]:
def align_model(learned_model, true_states, true_latents):

    posterior, _ = learned_model.infer(observations, num_iters=50)

    permutation = match_states(posterior.discrete.state_probs, true_states)
    learned_model = learned_model.permute(permutation)
    posterior = posterior.permute(permutation)

    inferred_latents = learned_model.latent_mean(posterior)

    alignment = align_latent(inferred_latents, true_latents)

    return learned_model.align(alignment)


matched_model = align_model(learned_model, true_states, true_latents)

posterior, _ = matched_model.infer(observations, num_iters=50)
inferred_states = matched_model.most_likely_states(posterior)
inferred_latents = matched_model.latent_mean(posterior)
reconstruction = matched_model.observation_mean(posterior)

In [ ]:
plot.plot_seq_1d_comparison(
    true_states,
    true_latents,
    inferred_states,
    inferred_latents,
)

plot.plot_seq_2d_comparison(
    true_states,
    true_latents,
    inferred_states,
    inferred_latents,
)

In [ ]:
reconstruction = matched_model.observation_mean(posterior)


plot.plot_traces_image_comparison(
    observations,
    reconstruction,
)

In [ ]:
plot.plot_dyn_conditional_linear_comparison(
    true_model.dynamics,
    matched_model.dynamics,
)